In [15]:
import glob
import os
import numpy as np
import xarray as xr
import geopandas as gpd
import regionmask
import warnings
import matplotlib.pyplot as plt

# Suppress specific xarray/dask warnings for cleaner output
warnings.filterwarnings("ignore")

# ==========================================
# Phase 1: Configuration Loader
# ==========================================
class PipelineConfig:
    """Centralized configuration manager for the pipeline."""
    def __init__(self, model_name, target_variable):
        self.model_name = model_name
        self.target_variable = target_variable

        #Hardcoding built-in paths
        self.base_path = "../CMIP6 data"
        self.shapefile_path = "../data/AmazonBasinLimits-master/amazon_sensulatissimo_gmm_v1.shp"
        
        # Default spatial bounding box for Amazon basin
        self.lat_min = -20
        self.lat_max = 10
        self.lon_min = 280
        self.lon_max = 320

    def get_dict(self):
        """Returns the configuration as a dictionary."""
        return {
            "model_name": self.model_name,
            "target_variable": self.target_variable,
            "base_path": self.base_path,
            "shapefile_path": self.shapefile_path,
            "lat_min": self.lat_min,
            "lat_max": self.lat_max,
            "lon_min": self.lon_min,
            "lon_max": self.lon_max
        }

# ==========================================
# Phase 2: Data Fetcher
# ==========================================
class DataFetcher:
    """Dynamically loads NetCDF files and handles safe time alignment."""
    def __init__(self, config):
        self.config = config
        self.model = config["model_name"]
        self.base_dir = config["base_path"]

    def _build_filepath(self, variable, experiment, frequency):
        """Constructs the file path pattern and verifies existence."""
        file_pattern = f"{variable}_{frequency}_{self.model}_{experiment}_r1i1p1f1_*.nc"
        full_path = os.path.join(self.base_dir, self.model, experiment, file_pattern)
        
        if not glob.glob(full_path):
            raise FileNotFoundError(f"No files found: {full_path}")
        return full_path

    def _safe_time_align(self, ds_target, ds_reference, target_name):
        """Validates and safely aligns time coordinates between Lmon and Amon."""
        if len(ds_target.time) != len(ds_reference.time):
            raise ValueError(f"Time length mismatch in {target_name}.")
            
        target_ym = ds_target.time.dt.strftime("%Y-%m").values
        ref_ym = ds_reference.time.dt.strftime("%Y-%m").values
        
        if not np.array_equal(target_ym, ref_ym):
            raise ValueError(f"Year-Month values do not match in {target_name}.")
            
        ds_target["time"] = ds_reference["time"]
        return ds_target

    def fetch_data(self):
        """Returns a dictionary of required raw datasets."""
        datasets = {}
        target_var = self.config["target_variable"]
        tas_files = glob.glob(self._build_filepath("tas", "1pctCO2", "Amon"))
        with xr.open_dataset(tas_files[0]) as ds_first:
            datasets["attrs_raw"] = ds_first.attrs.copy()

        # 1. Load temperature data for GWL
        datasets["tas_1pct"] = xr.open_mfdataset(self._build_filepath("tas", "1pctCO2", "Amon"))
        datasets["tas_pi"] = xr.open_mfdataset(self._build_filepath("tas", "piControl", "Amon"), combine="by_coords")

        # 2. Complex Variable Branch (MCWD and so on)
        if target_var == "MCWD":
            pr_ds = xr.open_mfdataset(self._build_filepath("pr", "1pctCO2", "Amon"))
            datasets["pr"] = pr_ds
            
            for var in ["evspsblsoi", "evspsblveg", "tran"]:
                var_ds = xr.open_mfdataset(self._build_filepath(var, "1pctCO2", "Lmon"))
                datasets[var] = self._safe_time_align(var_ds, pr_ds, var)
                
        # if target_var == "NEW_VARIABLE":
        #     pr_ds = xr.open_mfdataset(self._build_filepath("OTHER VARIABLE", "1pctCO2", "Amon or Lmon"))
        #     datasets["OTHER VARIABLE"] = VARIABL_NAME_ds

        # 3. Direct Variable Branch (Single Variable like cVeg)
        else:
            freq = "Amon" if target_var in ["tas", "pr"] else "Lmon"
            datasets["target_raw"] = xr.open_mfdataset(self._build_filepath(target_var, "1pctCO2", freq))

        return datasets

# ==========================================
# Phase 3: Core Processors
# ==========================================
class BaseProcessor:
    def __init__(self, config):
        self.config = config
        self.target_var = config["target_variable"]
        self.lat_slice = slice(config["lat_min"], config["lat_max"])
        self.lon_slice = slice(config["lon_min"], config["lon_max"])

    def process(self, datasets):
        raise NotImplementedError

class DirectVariableProcessor(BaseProcessor):
    """Processes simple variables by taking the annual mean."""
    def process(self, datasets):
        raw_da = datasets["target_raw"][self.target_var]
        original_attrs = raw_da.attrs.copy()
        sliced_da = datasets["target_raw"][self.target_var].sel(lat=self.lat_slice, lon=self.lon_slice)
        annual_da = sliced_da.resample(time="1YS").mean().load()
        annual_da.attrs = original_attrs
        annual_da.name = self.target_var
        return annual_da

class MCWDProcessor(BaseProcessor):
    """Processes the Maximum Cumulative Water Deficit (MCWD)."""
    def process(self, datasets):
        pr_amz = datasets["pr"]["pr"].sel(lat=self.lat_slice, lon=self.lon_slice)
        soi_amz = datasets["evspsblsoi"]["evspsblsoi"].sel(lat=self.lat_slice, lon=self.lon_slice)
        veg_amz = datasets["evspsblveg"]["evspsblveg"].sel(lat=self.lat_slice, lon=self.lon_slice)
        tran_amz = datasets["tran"]["tran"].sel(lat=self.lat_slice, lon=self.lon_slice)
        
        # Calculate Monthly ET and Water Deficit
        et_amz = soi_amz + veg_amz + tran_amz
        conversion_factor = pr_amz.time.dt.days_in_month * 86400
        wd_monthly = ((pr_amz - et_amz) * conversion_factor).load()
        
        # Cumulative Water Deficit Calculation
        cwd_values = np.zeros_like(wd_monthly.values)
        cwd_values[0] = np.minimum(0, wd_monthly.values[0])
        for t in range(1, len(wd_monthly.time)):
            cwd_values[t] = np.minimum(0, cwd_values[t-1] + wd_monthly.values[t])
            
        cwd_monthly = xr.DataArray(cwd_values, coords=wd_monthly.coords, dims=wd_monthly.dims)
        
        # Extract Annual Minimum (Maximum Deficit)
        mcwd_annual = cwd_monthly.resample(time="1YS").min().load()
        mcwd_annual.name = "MCWD"
        return mcwd_annual

class ProcessorFactory:
    """Routes data to the correct processor based on target variable."""
    @staticmethod
    def get_processor(config):
        if config["target_variable"] == "MCWD":
            return MCWDProcessor(config)
        return DirectVariableProcessor(config)

class GWLCalculator:
    """Computes the Global Warming Level anomaly."""
    @staticmethod
    def compute_gwl(tas_1pct_ds, tas_pi_ds):
        weights_1pct = np.cos(np.deg2rad(tas_1pct_ds["lat"]))
        tas_1pct_global = tas_1pct_ds["tas"].weighted(weights_1pct).mean(dim=["lat", "lon"])
        
        weights_pi = np.cos(np.deg2rad(tas_pi_ds["lat"]))
        tas_pi_global = tas_pi_ds["tas"].weighted(weights_pi).mean(dim=["lat", "lon"])
        
        tas_1pct_annual = tas_1pct_global.resample(time="1YS").mean().load()
        tas_pi_annual = tas_pi_global.resample(time="1YS").mean().load()
        
        tas_1pct_smoothed = tas_1pct_annual.rolling(time=10, center=True, min_periods=1).mean()
        tas_pi_smoothed = tas_pi_annual.rolling(time=10, center=True, min_periods=1).mean()
        
        gwl = tas_1pct_smoothed - tas_pi_smoothed
        gwl.name = "GWL"
        return gwl

# ==========================================
# Phase 4: Spatial Masker
# ==========================================
class SpatialMasker:
    """Applies the Amazon basin shapefile mask."""
    def __init__(self, config):
        self.shapefile_path = config["shapefile_path"]
        
    def apply_mask(self, data_array):
        amazon_boundary = gpd.read_file(self.shapefile_path)
        if amazon_boundary.crs is not None and amazon_boundary.crs.to_string() != "EPSG:4326":
            amazon_boundary = amazon_boundary.to_crs(epsg=4326)
        amazon_single_basin = amazon_boundary.unary_union
        
        lon_original = data_array["lon"]
        lon_shifted = data_array["lon"].where(data_array["lon"] <= 180, data_array["lon"] - 360)
        da_shifted = data_array.assign_coords(lon=lon_shifted)
        
        basin_region = regionmask.Regions([amazon_single_basin])
        mask = basin_region.mask(da_shifted["lon"], da_shifted["lat"], wrap_lon=False)
        
        masked_da_shifted = da_shifted.where(mask == 0)
        masked_da = masked_da_shifted.assign_coords(lon=lon_original)
        
        return masked_da

# ==========================================
# Phase 5: TOAD Formatter & Exporter
# ==========================================
class TOADExporter:
    """Formats and exports to a TOAD-compatible NetCDF file."""
    def __init__(self, config):
        self.model = config["model_name"]
        self.variable = config["target_variable"]

    def export(self, masked_da, gwl_da, raw_attrs):
        merged_ds = xr.merge([masked_da, gwl_da], join="inner")
        final_ds = merged_ds.set_coords("GWL").swap_dims({"time": "GWL"}).sortby("GWL")
        
        final_ds.attrs = raw_attrs
        final_ds.attrs["nominal_resolution"] = raw_attrs.get("nominal_resolution", "Not Available")

        vars_to_drop = [var for var in final_ds.data_vars if var != self.variable]
        final_ds = final_ds.drop_vars(vars_to_drop)
        
        output_dir = "./processed_data"
        os.makedirs(output_dir, exist_ok=True)
        output_filename = f"TOAD_{self.variable}_{self.model}.nc"
        output_path = os.path.join(output_dir, output_filename)
        
        final_ds.to_netcdf(output_path)
        print(f"[Export Success] Created NetCDF at: {output_path}")
        return output_path


# ==========================================
# Function 1: Generate and Save .nc file
# ==========================================
def generate_nc(model_name, target_variable):
    """Executes Phase 1 to 5 to process raw CMIP6 data and save it as a .nc file."""
    print(f"\n--- Generating .nc for {model_name} | {target_variable} ---")
    
    # Paths are now automatically handled inside PipelineConfig
    config_manager = PipelineConfig(model_name, target_variable)
    config = config_manager.get_dict()
    
    fetcher = DataFetcher(config)
    datasets = fetcher.fetch_data()
    
    processor = ProcessorFactory.get_processor(config)
    target_da = processor.process(datasets)
    gwl_da = GWLCalculator.compute_gwl(datasets["tas_1pct"], datasets["tas_pi"])
    
    masker = SpatialMasker(config)
    masked_target_da = masker.apply_mask(target_da)
    
    exporter = TOADExporter(config)
    output_nc_path = exporter.export(masked_target_da, gwl_da, datasets["attrs_raw"])
    
    return output_nc_path

# ==========================================
# Function 2: Load .nc file and Plot
# ==========================================
def plot_trend_from_nc(model_name, target_variable):
    """Loads a previously processed .nc file and generates a spatial mean plot."""
    file_path = f"./processed_data/TOAD_{target_variable}_{model_name}.nc"
    if not os.path.exists(file_path):
        print(f"[Error] File not found: {file_path}")
        return
        
    print(f"\n--- Plotting data from {file_path} ---")
    ds = xr.open_dataset(file_path)
    
    # Calculate Spatial Mean using lon and lat
    spatial_mean = ds[target_variable].mean(dim=["lon", "lat"], skipna=True)
        
    plot_metadata = {
        "cVeg": {"color": "forestgreen", "ylabel": "Mean Vegetation Carbon ($kgC/m^2$)"},
        "MCWD": {"color": "darkred", "ylabel": "Mean Max Cumulative Water Deficit (mm)"},
        "cSoil": {"color": "sienna", "ylabel": "Mean Soil Carbon ($kgC/m^2$)"},
        "treeFrac": {"color": "darkgreen", "ylabel": "Mean Tree Cover (%)"},
        "grassFrac": {"color": "limegreen", "ylabel": "Mean Grass Area (%)"},
        "baresoilFrac": {"color": "tan", "ylabel": "Mean Bare Soil Cover (%)"},
        "fVegLitter": {"color": "olive", "ylabel": "Mean Veg to Litter Flux ($kgC/m^2/s$)"},
        "fFire": {"color": "firebrick", "ylabel": "Mean Fire CO2 Emission Flux ($kgC/m^2/s$)"},
        "gpp": {"color": "teal", "ylabel": "Mean Gross Primary Production ($kgC/m^2/s$)"},
        "lai": {"color": "mediumseagreen", "ylabel": "Mean Leaf Area Index ($m^2/m^2$)"},
        "rGrowth": {"color": "purple", "ylabel": "Mean Autotrophic Respiration ($kgC/m^2/s$)"}
    }
    
    meta = plot_metadata.get(target_variable, {"color": "blue", "ylabel": f"Mean {target_variable}"})
    gwl_vals = spatial_mean["GWL"].values
    var_vals = spatial_mean.values
    
    plt.figure(figsize=(8, 5))
    plt.plot(gwl_vals, var_vals, color=meta["color"], linewidth=2.5, label=f"{model_name} {target_variable}")
    
    plt.title(f"Amazon Area-Averaged {target_variable} Response to GWL ({model_name})", fontsize=13)
    plt.xlabel("Global Warming Level (GWL) [°C]", fontsize=11)
    plt.ylabel(meta["ylabel"], fontsize=11)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()

    output_dir = "./plots"
    os.makedirs(output_dir, exist_ok=True)
    output_filename = f"TrendPlot_{target_variable}_{model_name}.png"
    output_path = os.path.join(output_dir, output_filename)
    
    plt.savefig(output_path, dpi=300)
    plt.close()
    
    print(f"[Plot Success] Saved graph at: {output_path}")

# ==========================================
# Function 3: Load .nc file and Plot Spatial Maps
# ==========================================
def plot_spatial_maps_from_nc(model_name, target_variable):
    """Loads a processed .nc file and generates a 1x6 grid of spatial maps."""
    file_path = f"./processed_data/TOAD_{target_variable}_{model_name}.nc"
    if not os.path.exists(file_path):
        print(f"[Error] File not found: {file_path}")
        return
        
    print(f"\n--- Plotting spatial maps from {file_path} ---")
    ds = xr.open_dataset(file_path)
    data_array = ds[target_variable]
    
    plot_metadata = {
        "cVeg": {"cmap": "YlGn", "label": "Vegetation Carbon ($kgC/m^2$)"},
        "MCWD": {"cmap": "OrRd", "label": "Max Cumulative Water Deficit (mm)"},
        "cSoil": {"cmap": "YlOrBr", "label": "Soil Carbon ($kgC/m^2$)"},
        "treeFrac": {"cmap": "Greens", "label": "Tree Cover (%)"},
        "grassFrac": {"cmap": "YlGn", "label": "Grass Area (%)"},
        "baresoilFrac": {"cmap": "Oranges", "label": "Bare Soil Cover (%)"},
        "fVegLitter": {"cmap": "YlGnBu", "label": "Veg to Litter Flux ($kgC/m^2/s$)"},
        "fFire": {"cmap": "Reds", "label": "Fire CO2 Emission ($kgC/m^2/s$)"},
        "gpp": {"cmap": "Greens", "label": "Gross Primary Production ($kgC/m^2/s$)"},
        "lai": {"cmap": "summer", "label": "Leaf Area Index ($m^2/m^2$)"},
        "rGrowth": {"cmap": "Purples", "label": "Autotrophic Respiration ($kgC/m^2/s$)"}
    }
    
    meta = plot_metadata.get(target_variable, {"cmap": "viridis", "label": f"{target_variable}"})

    target_gwls = [0.0, 1.5, 2.0, 3.0, 4.0, 5.0]
    v_min = float(data_array.min(skipna=True))
    v_max = float(data_array.max(skipna=True))

    # Built-in shapefile path
    shapefile_path = "../data/AmazonBasinLimits-master/amazon_sensulatissimo_gmm_v1.shp"
    amazon_boundary = gpd.read_file(shapefile_path)
    if amazon_boundary.crs is not None and amazon_boundary.crs.to_string() != "EPSG:4326":
        amazon_boundary = amazon_boundary.to_crs(epsg=4326)
    
    if data_array["lon"].max() > 180:
        amazon_boundary_plot = amazon_boundary.translate(xoff=360)
    else:
        amazon_boundary_plot = amazon_boundary

    fig, axes = plt.subplots(1, 6, figsize=(24, 4.5), sharex=True, sharey=True)

    for ax, target in zip(axes, target_gwls):
        da_slice = data_array.sel(GWL=target, method="nearest")
        actual_gwl = float(da_slice["GWL"])

        mesh = ax.pcolormesh(
            da_slice["lon"],
            da_slice["lat"],
            da_slice.values,
            cmap=meta["cmap"],
            vmin=v_min,
            vmax=v_max,
            shading="auto",
        )

        amazon_boundary_plot.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.5)
        ax.set_title(f"GWL: {target}°C\n(Actual: {actual_gwl:.2f}°C)", fontsize=11)
        ax.set_xlabel("Longitude [°E]", fontsize=9)

    axes[0].set_ylabel("Latitude [°N]", fontsize=10)

    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
    cbar = fig.colorbar(mesh, cax=cbar_ax)
    cbar.set_label(meta["label"], fontsize=11)

    plt.subplots_adjust(right=0.90)
    plt.suptitle(f"Amazon ${target_variable}$ Spatial Distribution by Global Warming Level ({model_name})", fontsize=14, y=1.05)
    
    output_dir = "./plots"
    os.makedirs(output_dir, exist_ok=True)
    output_filename = f"SpatialMap_{target_variable}_{model_name}.png"
    output_path = os.path.join(output_dir, output_filename)
    
    plt.savefig(output_path, bbox_inches="tight", dpi=300)
    plt.close()
    
    print(f"[Plot Success] Saved spatial map at: {output_path}")

In [17]:
# Target Configurations
MODEL = "NorCPM1"
VARIABLE = "cVeg"

# 1. Processing and saving .nc files
generate_nc(MODEL, VARIABLE)

# 2. Loading saved .nc file and plotting 1D Mean Line Graph
plot_trend_from_nc(MODEL, VARIABLE)

# 3. Loading saved .nc file and plotting 2D Spatial Maps
plot_spatial_maps_from_nc(MODEL, VARIABLE)


--- Generating .nc for NorCPM1 | cVeg ---
[Export Success] Created NetCDF at: ./processed_data/TOAD_cVeg_NorCPM1.nc

--- Plotting data from ./processed_data/TOAD_cVeg_NorCPM1.nc ---
[Plot Success] Saved graph at: ./plots/TrendPlot_cVeg_NorCPM1.png

--- Plotting spatial maps from ./processed_data/TOAD_cVeg_NorCPM1.nc ---
[Plot Success] Saved spatial map at: ./plots/SpatialMap_cVeg_NorCPM1.png
